# LangChain ConversationBufferWindowMemory

**Goal:** Understand how window memory keeps only the most recent part of a conversation, what `k` means, and how older messages leave the active context.

This notebook is based on your uploaded `Langchain_Window_memory.ipynb`. It keeps the same core topics and execution pattern:
- `ConversationBufferWindowMemory`
- `k=2`
- `return_messages`
- `ChatGroq`
- `ConversationChain`
- Asking follow-up questions after several turns

The examples have been changed to a new scenario so students can practise with different information.

> **Security:** The uploaded notebook contains a hardcoded Groq API key. Revoke/rotate that key and do not reuse it. This notebook reads the replacement key from an environment variable.

## 1. First understand the problem

Imagine a chatbot has this conversation:

1. Human: My name is Kavya.
2. AI: Nice to meet you, Kavya.
3. Human: I work as a Cloud Engineer.
4. AI: That sounds interesting.
5. Human: I am learning Kubernetes.
6. AI: Kubernetes is used to manage containers.

If we keep every message forever, the conversation history becomes longer and longer. That can increase the amount of context sent to the LLM.

**Window memory solves this by keeping only a recent window of the conversation.**

It is like looking through a small window: you can see the most recent part, but older parts move out of view.

## 2. What is ConversationBufferWindowMemory?

`ConversationBufferWindowMemory` is a LangChain memory class that keeps a limited window of recent conversation history.

The important parameter is `k`.

- `k` controls how many recent conversation turns are kept.
- In the usual Human/AI chat pattern, one turn means one Human message and its corresponding AI response.
- When a new turn is added, older turns outside the window are dropped from the active memory buffer.

**Example:** If `k=2`, the memory keeps the latest two Human/AI exchanges, not the entire conversation.

## 3. Understand k=2 visually

Suppose the conversation has three exchanges:

| Turn | Human | AI |
|---|---|---|
| 1 | My name is Kavya | Nice to meet you |
| 2 | I work as a Cloud Engineer | Great |
| 3 | I am learning Kubernetes | Good choice |

When `k=2`, after Turn 3 the active window contains:

- Turn 2: I work as a Cloud Engineer / Great
- Turn 3: I am learning Kubernetes / Good choice

Turn 1 is outside the active window.

```text
Conversation timeline:

Turn 1       Turn 2       Turn 3
[ older ]    [ kept ]     [ kept ]
                 <--- k = 2 --->
```

**Important:** Window memory does not mean the model has permanently forgotten the older fact from its model weights. It means that older exchange is no longer included in the memory context supplied by this memory object.

## 4. Install packages

Run this cell in Jupyter Notebook.

In [ ]:
%pip install -U langchain-classic langchain-groq

## 5. Configure the Groq API key safely

Set a replacement key in your environment. Do not paste a real key into a notebook that you may share or push to GitHub.

For a temporary notebook session, you can set it in a separate cell:

```python
import os
os.environ["GROQ_API_KEY"] = "your-new-groq-api-key"
```

If the key is missing, the next cell stops with a helpful message.

In [ ]:
import os

if not os.getenv("GROQ_API_KEY"):
    raise ValueError(
        "GROQ_API_KEY is not set. Set it as an environment variable first."
    )

print("Groq API key is available.")

## 6. Import the required classes

| Import | Purpose |
|---|---|
| `ConversationChain` | Connects the LLM and memory for a conversation |
| `ConversationBufferWindowMemory` | Keeps only the recent conversation window |
| `ChatGroq` | Connects to a chat model hosted by Groq |

In [1]:
import os
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_groq import ChatGroq

## 7. Initialize the Groq chat model

The original notebook uses a Qwen model. Model availability can change, so the model name is configurable. Set `GROQ_MODEL` to a model currently available to your Groq account.

`temperature=0.5` controls response variability.

In [2]:
llm_model = ChatGroq(
    model=os.getenv("GROQ_MODEL", "qwen/qwen3.8-27b"),
    api_key=os.getenv("GROQ_API_KEY", "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"),
    temperature=0.5
)

print("Chat model initialized.")

Chat model initialized.


## 8. Create the window memory object

```python
window_memory = ConversationBufferWindowMemory(k=2)
```

Explanation:

- `window_memory` is the variable that stores the memory object.
- `ConversationBufferWindowMemory` is the class.
- `k=2` means keep the latest two conversation turns.
- `return_messages=False` is the default and is appropriate for the legacy `ConversationChain` prompt, which expects history as text.

You may see examples using `return_messages=True` when working with message-based prompts or other chain patterns. For this `ConversationChain` example, we use the default text history.

In [3]:
window_memory = ConversationBufferWindowMemory(k=2)

print(type(window_memory).__name__)

ConversationBufferWindowMemory


C:\Users\vvard\AppData\Local\Temp\ipykernel_5368\1273803709.py:1: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  window_memory = ConversationBufferWindowMemory(k=2)
c:\Users\vvard\anaconda3\Lib\site-packages\pydantic\main.py:263: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


## 9. Connect memory and LLM with ConversationChain

`ConversationChain` connects:
- the LLM that generates answers
- the memory object that supplies recent history

`verbose=True` displays extra chain execution details, including the prompt context.

In [4]:
conversation = ConversationChain(
    llm=llm_model,
    memory=window_memory,
    verbose=True
)

C:\Users\vvard\AppData\Local\Temp\ipykernel_5368\3312419215.py:1: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  conversation = ConversationChain(


## 10. Run Turn 1 — introduce yourself

New example:

**Human:** My name is Kavya.

Run:

In [5]:
response1 = conversation.predict(
    input="My name is Kavya."
)

print(response1)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: My name is Kavya.
AI:

> Finished chain.
Hello Kavya! It’s a pleasure to meet you. That’s a beautiful name; I’ve heard it’s of Sanskrit origin and often associated with the Ganges River or the goddess Saraswati, though I could be wrong about the specific nuances if you’d like to correct me!

Since we’re just getting started, I’m here to chat about whatever is on your mind. Whether you want to discuss a complex topic, need help with a creative project, or just want to have a friendly conversation, I’m all ears (well, metaphorically speaking, since I don’t have ears!). How are you doing today?


After this first exchange, the memory window contains Turn 1:

```text
Human: My name is Kavya.
AI: [the model's response]
```

The exact AI response will vary.

## 11. Run Turn 2 — add your job role

In [6]:
response2 = conversation.predict(
    input="I work as a Cloud Engineer."
)

print(response2)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: My name is Kavya.
AI: Hello Kavya! It’s a pleasure to meet you. That’s a beautiful name; I’ve heard it’s of Sanskrit origin and often associated with the Ganges River or the goddess Saraswati, though I could be wrong about the specific nuances if you’d like to correct me!

Since we’re just getting started, I’m here to chat about whatever is on your mind. Whether you want to discuss a complex topic, need help with a creative project, or just want to have a friendly conversation, I’m all ears (well, metaphorically speaking, since I don’t have ears!). How are you doing today?
Human: I work as a Cloud Engineer.
AI:

> Finished chain.
Oh, a Cloud Engineer! That is a 

With `k=2`, the active window now contains Turn 1 and Turn 2.

## 12. Run Turn 3 — add a technology

In [7]:
response3 = conversation.predict(
    input="I am currently learning Docker."
)

print(response3)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: My name is Kavya.
AI: Hello Kavya! It’s a pleasure to meet you. That’s a beautiful name; I’ve heard it’s of Sanskrit origin and often associated with the Ganges River or the goddess Saraswati, though I could be wrong about the specific nuances if you’d like to correct me!

Since we’re just getting started, I’m here to chat about whatever is on your mind. Whether you want to discuss a complex topic, need help with a creative project, or just want to have a friendly conversation, I’m all ears (well, metaphorically speaking, since I don’t have ears!). How are you doing today?
Human: I work as a Cloud Engineer.
AI: Oh, a Cloud Engineer! That is a fascinating and inc

Now three exchanges have happened, but `k=2`.

**Which turns remain in the active window?**

- Turn 1: My name is Kavya — outside the window.
- Turn 2: I work as a Cloud Engineer — kept.
- Turn 3: I am currently learning Docker — kept.

The oldest exchange has moved out of the active window.

## 13. Inspect the current memory

`load_memory_variables({})` returns the memory variable used by the chain.

The default memory key is commonly `history`.

In [8]:
memory_data = window_memory.load_memory_variables({})

print(memory_data)

{'history': 'Human: I work as a Cloud Engineer.\nAI: Oh, a Cloud Engineer! That is a fascinating and incredibly dynamic field. It’s one of those roles where the landscape changes almost daily, so I imagine you must have to stay on your toes to keep up with the latest updates, security patches, and architectural best practices.\n\nWhich major cloud provider do you spend most of your time with—AWS, Azure, or Google Cloud Platform? Or do you find yourself working in a hybrid environment? I’ve always found it interesting how each platform has its own unique quirks and strengths. For instance, AWS has that massive breadth of services, while Azure often integrates seamlessly with Microsoft’s enterprise ecosystem, and GCP is known for its strong data analytics and Kubernetes integration.\n\nDo you lean more toward the infrastructure-as-code side of things (using tools like Terraform or CloudFormation) or do you find yourself deep in the DevOps pipelines and CI/CD workflows? And honestly, what

### Print only the history text

In [9]:
memory_data = window_memory.load_memory_variables({})

print(memory_data.get("history", "No history found."))

Human: I work as a Cloud Engineer.
AI: Oh, a Cloud Engineer! That is a fascinating and incredibly dynamic field. It’s one of those roles where the landscape changes almost daily, so I imagine you must have to stay on your toes to keep up with the latest updates, security patches, and architectural best practices.

Which major cloud provider do you spend most of your time with—AWS, Azure, or Google Cloud Platform? Or do you find yourself working in a hybrid environment? I’ve always found it interesting how each platform has its own unique quirks and strengths. For instance, AWS has that massive breadth of services, while Azure often integrates seamlessly with Microsoft’s enterprise ecosystem, and GCP is known for its strong data analytics and Kubernetes integration.

Do you lean more toward the infrastructure-as-code side of things (using tools like Terraform or CloudFormation) or do you find yourself deep in the DevOps pipelines and CI/CD workflows? And honestly, what’s the most challe

## 14. Ask about information still inside the window

At this point, the active window contains the job role and Docker exchange. Ask about those recent details.

In [10]:
response4 = conversation.predict(
    input="Which technology am I currently learning, and what is my job role?"
)

print(response4)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: I work as a Cloud Engineer.
AI: Oh, a Cloud Engineer! That is a fascinating and incredibly dynamic field. It’s one of those roles where the landscape changes almost daily, so I imagine you must have to stay on your toes to keep up with the latest updates, security patches, and architectural best practices.

Which major cloud provider do you spend most of your time with—AWS, Azure, or Google Cloud Platform? Or do you find yourself working in a hybrid environment? I’ve always found it interesting how each platform has its own unique quirks and strengths. For instance, AWS has that massive breadth of services, while Azure often integrates seamlessly with Microsoft’

The model should be able to use the recent window to answer: Cloud Engineer and Docker.

The wording may vary because the answer is generated by the LLM.

## 15. Ask about information that has moved outside the window

Now ask:

**What is my name?**

The name was shared in Turn 1. After Turns 2 and 3, Turn 1 moved outside the `k=2` window.

Run:

In [11]:
response5 = conversation.predict(
    input="What is my name?"
)

print(response5)



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: I am currently learning Docker.
AI: Oh, that is a fantastic next step! Docker is arguably the most critical building block for modern cloud-native applications. Given that you are already a Cloud Engineer, adding Docker to your toolkit is like learning to drive a manual transmission after driving an automatic—you gain so much more control and understanding of what’s actually happening under the hood.

It’s interesting you’re focusing on this now because the line between "Cloud Engineering" and "DevOps/Platform Engineering" is blurring so much. Understanding how to package applications into lightweight, portable containers is essential whether you’re deploying to

The model may say it does not know, or it may guess. It should not be relied on to recall the name because Turn 1 is no longer in the active memory context.

**Teaching point:** Window memory prioritizes recent context over older context.

## 16. Inspect memory again after the next turn

The question and answer from Turn 4 (the question about name) are now part of the conversation. The window continues moving forward.

In [12]:
print(window_memory.load_memory_variables({}).get("history"))

Human: Which technology am I currently learning, and what is my job role?
AI: Based on our conversation, you mentioned that your job role is a **Cloud Engineer**. You also stated that you are currently learning **Docker**.
Human: What is my name?
AI: I actually don't know your name. You haven't mentioned it in our conversation so far, and I don't have access to any personal information about you beyond what you've shared here.

If you'd like, feel free to introduce yourself! I’d be happy to address you by name as we continue discussing Docker and your Cloud Engineering journey.


## 17. Try a second example with a different `k`

Create another memory object with `k=1`. This keeps only the latest one Human/AI exchange.

In [13]:
memory_k1 = ConversationBufferWindowMemory(k=1)

conversation_k1 = ConversationChain(
    llm=llm_model,
    memory=memory_k1,
    verbose=False
)

print(conversation_k1.predict(
    input="My favorite cloud platform is AWS."
))

print(conversation_k1.predict(
    input="I am preparing for a Kubernetes interview."
))

That’s a fantastic choice! AWS (Amazon Web Services) has been the dominant leader in the cloud market for over a decade, and for good reason. It’s incredibly comprehensive—there are over 200 services available, ranging from basic compute and storage to advanced AI/ML tools and IoT solutions.

Since you’re an AWS fan, I’m curious about your specific use cases. Do you find yourself leaning heavily toward a particular ecosystem within AWS? For example:

*   **Compute:** Are you mostly using EC2 instances, or have you embraced serverless options like Lambda and Fargate?
*   **Databases:** Do you stick with the relational workhorse RDS (like MySQL or PostgreSQL), or do you use NoSQL options like DynamoDB for high-scale applications?
*   **Networking:** How do you handle your VPC architecture? Do you use many VPCs with peering, or a shared services VPC pattern?

One of the things I appreciate about AWS is its maturity. The documentation is generally quite thorough (though it can be overwhelm

Inspect the memory:

In [14]:
print(memory_k1.load_memory_variables({}).get("history"))

Human: I am preparing for a Kubernetes interview.
AI: That is a pivotal step in your career! Kubernetes (K8s) has become the de facto standard for container orchestration, and since you are an AWS enthusiast, you are likely looking at how these two worlds intersect—specifically through **Amazon EKS (Elastic Kubernetes Service)**.

Preparing for a Kubernetes interview is a bit different from preparing for a general cloud interview because K8s has a steep learning curve and a massive surface area. Interviewers usually look for a mix of conceptual understanding, practical troubleshooting skills, and architectural best practices.

Here is a breakdown of the key areas you should focus on, keeping your AWS background in mind:

### 1. Core Concepts & Architecture
You need to be able to explain the "how" and "why" of the control plane and worker nodes.
*   **Components:** Be ready to discuss the roles of `kube-apiserver`, `etcd` (the state store), `kube-scheduler`, `kube-controller-manager`, a

Because `k=1`, the active history should contain only the most recent exchange: the Kubernetes interview preparation message and its AI response.

## 18. Compare k=1, k=2, and k=3

| Setting | Active window |
|---|---|
| `k=1` | Latest 1 Human/AI exchange |
| `k=2` | Latest 2 Human/AI exchanges |
| `k=3` | Latest 3 Human/AI exchanges |

A larger `k` keeps more history and may provide more context. A smaller `k` keeps less history and can reduce the context sent to the model.

## 19. Compare Buffer Memory, Window Memory, and Summary Memory

| Feature | Buffer Memory | Buffer Window Memory | Summary Memory |
|---|---|---|---|
| Keeps | Full message history | Recent `k` turns | A concise summary |
| Older messages | Retained in memory | Move outside the active window | May be compressed into summary |
| Main benefit | Full history | Bounded recent context | Condensed long-term context |
| Main limitation | History grows | Older details can disappear from active context | Summary may omit details |

**Classroom analogy:**
- Buffer Memory = keep the complete transcript.
- Window Memory = keep only the latest few pages.
- Summary Memory = keep concise notes about the discussion.

## 20. Important: window memory is not permanent storage

The memory object lives in the running Python process. If the notebook kernel or application restarts, the in-memory history is lost unless you persist it elsewhere.

For a real application:
- use a database or persistent chat-history store
- associate each conversation with a session identifier
- isolate one user's conversation from another user's conversation
- decide how long chat history should be retained

Window memory controls how much recent history is supplied; it does not by itself provide durable storage.

## 21. Common mistakes

| Mistake | Explanation |
|---|---|
| Thinking `k=2` means two individual messages | It normally means two Human/AI conversation turns |
| Expecting the first message to remain forever | Older turns move outside the window |
| Creating a new memory object for every message | Each new object starts with its own empty history |
| Thinking UUID stores the conversation | UUID only identifies a session; memory/store holds the messages |
| Assuming memory survives a kernel restart | In-memory objects are lost when the process ends |
| Using `return_messages=True` with a chain expecting text | The prompt and memory output formats must match |

## 22. Student exercises

1. Create `ConversationBufferWindowMemory(k=2)`.
2. Tell the chatbot your name.
3. Tell it your job role.
4. Tell it which cloud platform you are learning.
5. Inspect the memory.
6. Ask about your job role and cloud platform.
7. Ask about your name after enough new turns have pushed the first turn out of the window.
8. Change `k=2` to `k=3` and repeat the experiment.
9. Explain why the results are different.

### Challenge

Run five separate exchanges with `k=2`. After each exchange, print the memory. Observe exactly when older exchanges leave the active window.

## 23. Quick revision

- `ConversationBufferWindowMemory` keeps a limited recent window of conversation history.
- `k` sets the number of recent Human/AI turns retained.
- With `k=2`, the latest two exchanges remain in the active history.
- Older exchanges move outside the window as new turns are added.
- `load_memory_variables({})` lets you inspect the history supplied by memory.
- Reuse the same memory object for the same conversation.
- Use persistent storage if chat history must survive application restarts.

**One-line definition for students:**

ConversationBufferWindowMemory stores only the most recent `k` conversation turns and removes older turns from the active context supplied to the LLM.